In [3]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


In [4]:
#DATA_PATH = "/home/denklewer/Data/gbm122/pVACseq/"
DATA_PATH = "D:/Google Drive/Work/ITMO/data/gbm122/pVACseq/"
METHOD = "mhcattnnet"

In [5]:
from data_reading import read_gbm122_data

per_region_total_df , per_region_per_allele_mt_df, per_region_per_allele_wt_df = read_gbm122_data(DATA_PATH)


Region 1, unique (Transcrpit, Epitope pairs): 13197
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-B*35:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-B*40:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-C*03:04: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-C*04:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
Region 2, unique (Transcrpit, Epitope pairs): 13273
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13279
take unique (MT_epitope, WT_epitope) pairs
result length 5127
allele HLA-B*35:01: initial length 13279
take unique (MT_epitope, WT_epitope) pairs
result length 51

D:\Tools\Python\Anaconda\envs\mhcattnnet\lib\site-packages\IPython\core\interactiveshell.py:2818: DtypeWarning: Columns (10) have mixed types.Specify dtype option on import or set low_memory=False.
  if self.run_code(code, result):


Region 3, unique (Transcrpit, Epitope pairs): 15170
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-B*35:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-B*40:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-C*03:04: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-C*04:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
Region 4, unique (Transcrpit, Epitope pairs): 13757
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13757
take unique (MT_epitope, WT_epitope) pairs
result length 4932
allele HLA-B*35:01: initial length 13757
take unique (MT_epitope, WT_epitope) pairs
result length 49

# 0. Preparations

In [6]:
sample_region_dict = next(iter(per_region_per_allele_wt_df.values()))
sample_allele_df =  next(iter(sample_region_dict.values()))

ALLELES = list(sample_region_dict.keys())
REGIONS = list(per_region_total_df.keys())
ALLELES, REGIONS

(['HLA-A*03:01', 'HLA-B*35:01', 'HLA-B*40:01', 'HLA-C*03:04', 'HLA-C*04:01'],
 [1, 2, 3, 4, 5])

## 0.1. make result dicts

In [7]:
per_region_per_method_per_allele_wt_predictions = dict()
per_region_per_method_per_allele_mt_predictions = dict()
for region in per_region_total_df.keys():
    per_region_per_method_per_allele_wt_predictions[region] = dict()
    per_region_per_method_per_allele_mt_predictions[region] = dict()

# MHCAttnNetMethods

In [8]:
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import pandas as pd
import os
import sys
import numpy as np
#PREDICTOR_HOME = os.getenv('MHCATTNNET_HOME') # environment variable defined
PREDICTOR_HOME = "D:\Projects\MHCAttnNet" # my windows PCs

In [9]:
sys.path.append(PREDICTOR_HOME + '/src')

from alleles_I import c1_map
from alleles_II import c2_map
PREDICTOR_HOME

'D:\\Projects\\MHCAttnNet'

In [42]:
import shutil
from pathlib import Path
CLASSES_TO_USE = ['I']
def defineClass(allele_name):
    return "I"


def add_mhc_sequence_column(df, allele_name):
    if defineClass(allele_name) == 'I':
        df['mhc_amino_acid'] = df.mhc_allele.apply(lambda x: c1_map.get(x, x))
    else:
        df['mhc_amino_acid'] =  df.mhc_allele.apply(lambda x: c2_map.get(x, x))
            
def reorder_columns(per_class_df):
    for class_to_use,df in per_class_df.items():
        df = df[['peptide', 'mhc_amino_acid', 'bind', 'mhc_allele', 'source_sequence_name']]
        per_class_df[class_to_use] = df

def make_per_class_dfs(per_allele_df):
    per_class_dfs = dict()
    for class_to_use in CLASSES_TO_USE:
        per_class_dfs[class_to_use] = list()
    
    per_class_single_df = dict()
    for allele_name in ALLELES:
        target_allele = normalize_allele_name(allele_name).replace('*','')
        df = per_allele_df[allele_name].copy()
        df['mhc_allele'] = target_allele
        df['bind'] = 1
        add_mhc_sequence_column(df, allele_name)

        assert 'peptide' in df
        assert 'mhc_allele' in df
        assert 'bind' in df #dummy value 
        assert 'source_sequence_name' in df
        assert 'mhc_amino_acid' in df
        per_class_dfs[defineClass(allele_name)].append(df)
        
    for class_to_use, list_of_dfs in per_class_dfs.items():
        per_class_single_df[class_to_use] = pd.concat(list_of_dfs)
         
    reorder_columns(per_class_single_df)
    return per_class_single_df


        
def filter_df(df):
    result_df = df[df['peptide'].notnull()].copy()
        
    return result_df
        
def save_preprocessed_dfs(per_class_df, mode='val'):
    for class_to_use in CLASSES_TO_USE:
        
        target_data_base_path = custom_config.mhc_1_base_path if class_to_use == 'I' else custom_config.mhc_2_base_path
        Path(target_data_base_path).mkdir(parents=True, exist_ok=True)
        
#     per_class_test_df[class_to_use].to_csv(f"{PATH_TO_DATA}/class{class_to_use}/test.csv",index=False)
        filter_df(per_class_df[class_to_use]).to_csv(f"{target_data_base_path}/{mode}.csv",index=False)

    

# 1. Train Default MHCAttnNet

In [11]:
import mhcattnnet_config as custom_config

In [12]:
# save train data
DEFAULT_TRAIN_DATA_PATH = "D:/Sandbox/dataset/"
per_allele_train_data = dict()
for class_to_use in CLASSES_TO_USE:
    per_allele_train_data[class_to_use] = pd.read_csv("{}class{}/train.csv".format(DEFAULT_TRAIN_DATA_PATH, class_to_use))
    #per_allele_train_data[class_to_use] [RESULT_COLUMNS['id']] =  'seq' + per_allele_train_data[class_to_use] .index.astype(str).values

save_preprocessed_dfs(per_allele_train_data, mode='train')    



In [ ]:
from train import ModelTrainer

custom_config.early_stop = False
custom_config.read_all_columns = False

trainer_object = ModelTrainer(custom_config=custom_config)
for class_to_use in CLASSES_TO_USE:
    numeric_class = 1 if class_to_use=='I' else 2
    trainer_object.train_models(mhc_class=numeric_class)
    
# Done!

# 1. Predict with MHCAttnNet

In [43]:
from tqdm import tqdm
import torch
from model import MHCAttnNet

from data_loader import DataLoader

In [87]:
def run_validation_for_class(class_to_use):
    custom_config.read_all_columns = True
    if class_to_use == 'I':
        custom_config.model_name = custom_config.mhc_1_model_name
        custom_config.base_path = custom_config.mhc_1_base_path
    else:
        custom_config.model_name = custom_config.mhc_2_model_name
        custom_config.base_path = custom_config.mhc_2_base_path
    device = custom_config.device
    data_loader_obj  = DataLoader(custom_config)
    train_loader, val_loader, peptide_embedding, mhc_embedding = data_loader_obj.get_dataset(device, shuffle=False)
    ## Read model
    total_result = None
    num_models = custom_config.num_models
    for model_index in range(num_models):
        model = MHCAttnNet(peptide_embedding, mhc_embedding, custom_config)
        model.load_state_dict(torch.load(custom_config.model_name + "_" + str(model_index)))
        model.to(device)
        y_actual_val = []
        y_pred_val = []
        seq_id = []
        mhc_allele = []
        for row in tqdm(val_loader):
            y_pred = model.forward(row.peptide, row.mhc_amino_acid)
            y_actual = row.bind
            y_actual_val.extend(list(torch.squeeze(y_actual).cpu().data.numpy()))
            y_pred_val.extend(list(torch.squeeze(y_pred).cpu().data.numpy()))
            seq_id.extend(row.source_sequence_name)
            mhc_allele.extend(row.mhc_allele)

        result_df = pd.DataFrame({"source_sequence_name":seq_id, "binder":y_actual_val, f"score_{model_index}": y_pred_val, "mhc_allele":mhc_allele} )
        if model_index == 0:
            total_result = result_df
        else:
            total_result = total_result.merge(result_df, on=['source_sequence_name', 'binder', 'mhc_allele'])
    total_result['score'] = total_result[[f"score_{i}" for i in range(num_models)]].median(axis=1)
    total_result = total_result.rename(columns={"mhc_allele":"allele"})
    return total_result


def update_with_predictions_and_split_by_allele(per_allele_source_df, per_class_predictions):
    per_allele_predictions = dict()  
    for allele_name in per_allele_source_df.keys():
        class_to_use = defineClass(allele_name)
        target_allele = normalize_allele_name(allele_name).replace('*','')
        result_df = per_allele_source_df[allele_name].copy()
        result_df['score'] = np.nan
        pred_df = per_class_predictions[class_to_use]
        predictions = pred_df.loc[pred_df['allele'] == target_allele,'score'].values
        result_df.loc[result_df['peptide'].notnull(), 'score'] = predictions
        per_allele_predictions[allele_name] = result_df
    
        
    return per_allele_predictions


def predict_with_mhcattnnet(per_allele_df):
    per_class_df = make_per_class_dfs(per_allele_df)
    save_preprocessed_dfs(per_class_df)
    per_class_result = dict()
    for class_to_use in CLASSES_TO_USE:
        per_class_result[class_to_use] =  run_validation_for_class(class_to_use)
    per_allele_result = update_with_predictions_and_split_by_allele(per_allele_df,per_class_result)
    return per_allele_result
    

In [94]:
for region in REGIONS:
    print("region {}".format(region))
    print("\t mt")
    per_allele_df = per_region_per_allele_mt_df[region]
    per_region_per_method_per_allele_mt_predictions[region][METHOD] = predict_with_mhcattnnet(per_allele_df)  

    print("\t wt")
    per_allele_df = per_region_per_allele_wt_df[region]
    per_region_per_method_per_allele_wt_predictions[region][METHOD] = predict_with_mhcattnnet(per_allele_df)  


region 1
	 mt


  0%|                                                                                           | 0/53 [00:00<?, ?it/s]

Peptide embedding dimension torch.Size([23, 100])
MHC Amino Acid embedding dimension torch.Size([374, 100])


100%|██████████████████████████████████████████████████████████████████████████████████| 53/53 [00:02<00:00, 18.78it/s]


	 wt


  0%|                                                                                           | 0/37 [00:00<?, ?it/s]

... (30 outputs left out)

	 wt


  0%|                                                                                           | 0/44 [00:00<?, ?it/s]

Peptide embedding dimension torch.Size([23, 100])
MHC Amino Acid embedding dimension torch.Size([374, 100])


100%|██████████████████████████████████████████████████████████████████████████████████| 44/44 [00:02<00:00, 18.31it/s]


# 2. Join predictions

In [95]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':['source_sequence_name', 'score', 'affinity'],
    'acme':['source_sequence_name','score', ],
    'puffin':['source_sequence_name', 'score', 'epistemic_var', 'aleatoric_var', 'binding_likelihood'],
    'mhcseqnet':['source_sequence_name', 'score'],
    'mhcattnnet':['source_sequence_name', 'score']
}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df
    

def getPredictionResult(total_results, region, allele_name, method):
    df = total_results[region][method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df


In [96]:
def joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions):
    per_region_per_allele_result = dict()
    for region in REGIONS:
        print("region", region)
        per_region_per_allele_result[region] = dict()
        methods = list(per_region_per_method_per_allele_mt_predictions[region].keys())
        for allele_name in ALLELES:
            print("allele", allele_name)
            df_source_wt = per_region_per_allele_wt_df[region][allele_name].copy()
            df_source_mt = per_region_per_allele_mt_df[region][allele_name].copy()
            
            for method in methods:
                print("method", method)
                df_method_wt = getPredictionResult(per_region_per_method_per_allele_wt_predictions, region, allele_name, method)
                df_method_mt = getPredictionResult(per_region_per_method_per_allele_mt_predictions, region, allele_name, method)
                df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
                df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
            leave_columns = [ 'source_sequence_name', 'allele', 'length']
            df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
            df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
            # rename id back 

            
            df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
            per_region_per_allele_result[region][allele_name] = df_result
    return per_region_per_allele_result
                
                

              

In [97]:
per_region_per_allele_result = joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions)

region 1
allele HLA-A*03:01
method mhcattnnet
allele HLA-B*35:01
method mhcattnnet
allele HLA-B*40:01
method mhcattnnet
allele HLA-C*03:04
method mhcattnnet
allele HLA-C*04:01
method mhcattnnet
region 2
allele HLA-A*03:01
method mhcattnnet
allele HLA-B*35:01
method mhcattnnet
allele HLA-B*40:01
method mhcattnnet
allele HLA-C*03:04
method mhcattnnet
allele HLA-C*04:01
method mhcattnnet
region 3
allele HLA-A*03:01
method mhcattnnet
allele HLA-B*35:01
method mhcattnnet
allele HLA-B*40:01
method mhcattnnet
allele HLA-C*03:04
method mhcattnnet
allele HLA-C*04:01
method mhcattnnet
region 4
allele HLA-A*03:01
method mhcattnnet
allele HLA-B*35:01
method mhcattnnet
allele HLA-B*40:01
method mhcattnnet
allele HLA-C*03:04
method mhcattnnet
allele HLA-C*04:01
method mhcattnnet
region 5
allele HLA-A*03:01
method mhcattnnet
allele HLA-B*35:01
method mhcattnnet
allele HLA-B*40:01
method mhcattnnet
allele HLA-C*03:04
method mhcattnnet
allele HLA-C*04:01
method mhcattnnet


# 5. For now just write to files

In [98]:
#DATA_PATH = "/home/denklewer/Data/gbm122/" #ubuntu machine
DATA_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/gbm122/" # windows

TARGET_PATH_TO_PREDICTIONS = DATA_PATH + '{}/'.format(METHOD)

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [99]:
# for decoy_key in DECOY_PREFIXES:

for region in REGIONS:
    for allele_name in ALLELES:
        total_df = per_region_per_allele_result[region][allele_name]
        allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        total_df.to_csv("{}{}_predictions_reg_{}_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,region,allele_name),sep=";", index=False)

In [100]:
total_df

,allele,length,peptide_mt,source_sequence_name,mhcattnnet_score_mt,peptide_wt,mhcattnnet_score_wt
0,HLA-C*04:01,8,TPRTRNKF,seq14936,1.000000,PPRTRNKF,0.999947
1,HLA-C*04:01,8,SWPGAPVY,seq14939,1.000000,SWPGTPVY,0.996648
2,HLA-C*04:01,8,YVAVCHPL,seq14940,0.999977,YVAVCRPL,0.991328
3,HLA-C*04:01,8,RELLSLAF,seq14941,0.999993,RGLLSLAF,0.996577
4,HLA-C*04:01,8,NQYASVAL,seq14942,0.999999,NaN,NaN
...,...,...,...,...,...,...,...
4942,HLA-C*04:01,11,LHGTLQKDGWT,seq65802,1.000000,LRGTLQKDGWT,0.999987
4943,HLA-C*04:01,11,SSSSSSSNNNY,seq65806,0.988417,SSSSSSNNNNY,0.327243
4944,HLA-C*04:01,11,RSSSSSSSSSN,seq65807,0.999985,RSSSSSSSSNN,0.180012
4945,HLA-C*04:01,11,SSSSSSSSSNN,seq65808,0.999995,SSSSSSSSNNN,0.032064
